# G1 Academy Bonus - Task 4: robot-state observation (dictionary-based get_* interface)

## Introduction
This task builds the dictionary-based state readers described in `notes.txt` section 2: `get_lowstate`, `get_odommodestate`, `get_battery`, `get_slam_info`, `get_occupancygrid`, `get_rgbd`, and `get_services`. Each one wraps a native subscriber or SDK client and normalizes the result into a plain dictionary, so higher-level tasks never touch raw DDS message layouts directly. This notebook is self-contained: it repeats the `ensure_channel_factory`/`Latest` helpers from Task 2 so it can run on its own.

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

## Task 1 - `get_lowstate()`
Joint positions/velocities/torques and IMU fields, read from the cached `rt/lowstate` message.

In [ ]:
lowstate_sub = Latest("rt/lowstate", LowState_)

def get_lowstate():
    msg = lowstate_sub.message
    if msg is None:
        return None
    motors = list(msg.motor_state)
    imu = msg.imu_state
    return {
        "timestamp": lowstate_sub.timestamp,
        "joint_positions": [float(m.q) for m in motors],
        "joint_velocities": [float(m.dq) for m in motors],
        "joint_torques": [float(m.tau_est) for m in motors],
        "imu": {
            "rpy": [float(imu.rpy[i]) for i in range(3)],
            "gyro": [float(imu.gyroscope[i]) for i in range(3)],
            "acc": [float(imu.accelerometer[i]) for i in range(3)],
        },
    }

# print(get_lowstate())

## Task 2 - `get_odommodestate()`
Field names on `SportModeState_` vary a little across SDK/firmware versions, so read defensively with `getattr` fallbacks instead of assuming one exact attribute name - the same defensive pattern `sdk_wrapper` uses internally.

In [ ]:
from unitree_sdk2py.idl.unitree_go.msg.dds_ import SportModeState_

odom_sub = Latest("rt/odommodestate", SportModeState_)

def _first_attr(obj, names, default=None):
    for name in names:
        if hasattr(obj, name):
            return getattr(obj, name)
    return default

def get_odommodestate():
    msg = odom_sub.message
    if msg is None:
        return None
    position = _first_attr(msg, ("position", "pos", "position_w"))
    velocity = _first_attr(msg, ("velocity", "vel"))
    gait = _first_attr(msg, ("gait_type", "gaitType", "gait"))
    return {
        "timestamp": odom_sub.timestamp,
        "position": None if position is None else tuple(float(x) for x in position),
        "velocity": None if velocity is None else tuple(float(x) for x in velocity),
        "mode": None if getattr(msg, "mode", None) is None else int(msg.mode),
        "gait_type": None if gait is None else int(gait),
    }

# print(get_odommodestate())

## Task 3 - `get_battery()`
Battery data can arrive on a dedicated `BmsState_` topic (name and message module vary by platform), or embedded inside `rt/lowstate`. Try the dedicated topics first, fall back to `rt/lowstate` fields.

In [ ]:
import importlib

def _bms_types():
    types = []
    for module_name in ("unitree_sdk2py.idl.unitree_hg.msg.dds_", "unitree_sdk2py.idl.unitree_go.msg.dds_"):
        module = importlib.import_module(module_name)
        if hasattr(module, "BmsState_"):
            types.append(module.BmsState_)
    return types

BMS_TOPICS = ["rt/lf/bmsstate", "rt/lf/agvbmsstate", "rt/bmsstate", "rt/agvbmsstate"]
bms_subs = [(topic, Latest(topic, msg_type)) for topic in BMS_TOPICS for msg_type in _bms_types()]

def get_battery():
    for topic, sub in bms_subs:
        if sub.fresh(max_age_s=3.0):
            msg = sub.message
            return {"source": topic, "timestamp": sub.timestamp, "soc": int(msg.soc), "current": int(msg.current), "cycle": int(msg.cycle)}
    msg = lowstate_sub.message
    if msg is None:
        return None
    return {
        "source": "rt/lowstate",
        "timestamp": lowstate_sub.timestamp,
        "power_v": None if getattr(msg, "power_v", None) is None else float(msg.power_v),
        "power_a": None if getattr(msg, "power_a", None) is None else float(msg.power_a),
    }

# print(get_battery())

## Task 4 - `get_slam_info()`
SLAM status arrives as a JSON-in-`String_` payload on `rt/slam_info`, with `rt/slam_key_info` as a fallback.

In [ ]:
from unitree_sdk2py.idl.std_msgs.msg.dds_ import String_

slam_info_sub = Latest("rt/slam_info", String_)
slam_key_sub = Latest("rt/slam_key_info", String_)

def get_slam_info():
    if slam_info_sub.message is not None:
        return slam_info_sub.message.data
    if slam_key_sub.message is not None:
        return slam_key_sub.message.data
    return None

# print(get_slam_info())

## Task 5 - `get_occupancygrid()`: a documented limitation
`academy/todo.txt` flags this explicitly: *"Add a direct occupancy-grid subscriber once the deployed topic and message type are confirmed."* Guessing a topic/type pair and silently returning wrong or empty data is worse than refusing. Confirm the live topic first (for example `ros2 topic list` / `ros2 topic info -v` against the running SLAM stack), then wire a `Latest` subscriber the same way `get_slam_info()` does above.

In [ ]:
def get_occupancygrid():
    raise NotImplementedError(
        "Confirm the deployed occupancy-grid topic/message type against the running SLAM stack "
        "before implementing this (see academy/todo.txt); do not guess a topic/type pair."
    )

## Task 6 - `get_rgbd()`
RGB-D frames come from `rgbd_server_service` (see `realsense_rgbd_zmq_stream.py`), a ZMQ PUB socket sending `[rgb_jpeg, depth_png, depth_scale]` multipart frames. Connect as SUB, take the newest frame, and decode the pieces lazily (callers decide whether they need the depth channel).

In [ ]:
import struct

def get_rgbd(endpoints=("tcp://127.0.0.1:5555", "tcp://localhost:5555")):
    import zmq
    ctx = zmq.Context.instance()
    last_error = None
    for endpoint in endpoints:
        sock = ctx.socket(zmq.SUB)
        sock.setsockopt(zmq.SUBSCRIBE, b"")
        sock.setsockopt(zmq.RCVTIMEO, 3000)
        try:
            sock.connect(endpoint)
            parts = sock.recv_multipart()
            if len(parts) < 3:
                continue
            scale = struct.unpack("f", parts[2])[0] if parts[2] != b"0" and len(parts[2]) == 4 else None
            return {
                "timestamp": time.time(),
                "endpoint": endpoint,
                "rgb_jpeg": bytes(parts[0]),
                "depth_png": None if parts[1] == b"0" else bytes(parts[1]),
                "depth_scale": scale,
            }
        except Exception as exc:
            last_error = exc
        finally:
            sock.close(0)
    if last_error is not None:
        raise last_error
    return None

# frame = get_rgbd()

## Task 7 - `get_services()`
`RobotStateClient.ServiceList()` returns every registered service's name/status/protect flag; annotate the known ones with the documented description catalog.

In [ ]:
try:
    from unitree_sdk2py.b2.robot_state.robot_state_client import RobotStateClient
except ImportError:
    from unitree_sdk2py.go2.robot_state.robot_state_client import RobotStateClient

robot_state_client = RobotStateClient()
robot_state_client.SetTimeout(5.0)
robot_state_client.Init()

SERVICE_CATALOG = {
    "ai_sport": "Main Motion Control Service", "basic_service": "Basic Service",
    "g1_arm_example": "Upper Limb Motion Service", "vui_service": "Audio and Lighting Control Service",
    "unitree_slam": "Navigation Service",
}

def get_services():
    code, service_states = robot_state_client.ServiceList()
    if int(code) != 0:
        raise RuntimeError(f"ServiceList failed: {code}")
    return [
        {"name": s.name, "description": SERVICE_CATALOG.get(s.name, ""), "status": int(s.status), "protected": bool(s.protect)}
        for s in service_states
    ]

# print(get_services())

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.